# 07 · Predição — quem vai sair nos próximos 28 dias

Fluxo de produção: lê a safra mais recente da **feature store**, junta os quatro blocos, carrega o **modelo registrado no MLflow** (Unity Catalog) e ranqueia os usuários pela probabilidade de churn.

Não treina nada: usa o modelo que o `06_avaliacao` registrou com o alias `champion`. Para pontuar um mês novo, basta gravar a safra no `02_feature_store` e rodar este notebook.

**Entrada:** `fs_transacoes`, `fs_produtos`, `fs_lives`, `fs_cursos` e o modelo `MODELO_REGISTRADO@champion` · **Saída:** tabela `predicoes_churn` (histórico de todas as safras pontuadas) e `output/tables/top50_churn_<safra>.csv`

In [0]:
%pip install feature_engine xgboost lightgbm -q

In [0]:
%run ./00_setup

## 1. Safra a pontuar
Por padrão, a safra mais recente da feature store. Para outra, troque `DT_PREDICAO` (ex.: `'2026-10-01'`).

In [0]:
#DT_PREDICAO = spark.sql(f'SELECT MAX(dtRef) FROM {FS_TRANSACOES}').first()[0].strftime('%Y-%m-%d')
DT_PREDICAO = '2026-07-01'

print(f'Safra a pontuar: {DT_PREDICAO} ({mes(DT_PREDICAO, True)}) · churn nos 28 dias a partir dessa data')

## 2. Features da safra
Junta os quatro blocos da feature store, como no `02b_abt`, mas só para a safra a pontuar e sem o alvo (ainda não aconteceu).

In [0]:
dados = None
for tabela in [FS_TRANSACOES, FS_PRODUTOS, FS_LIVES, FS_CURSOS]:
    bloco = spark.table(tabela).filter(f"dtRef = DATE'{DT_PREDICAO}'").toPandas()
    dados = bloco if dados is None else dados.merge(bloco, on=['dtRef', 'IdCliente'], how='left')

assert len(dados) > 0, f'Safra {DT_PREDICAO} não está na feature store: rode o 02a_feature_store para essa data'
print(f'{len(dados):,} usuários ativos na safra'.replace(',', '.'))

## 3. Modelo registrado no MLflow
Carrega a versão marcada como `champion`. As features e os tipos vêm da assinatura salva com o modelo, então a entrada fica igual à do treino.

In [0]:
import os, logging
os.environ['MLFLOW_DATABRICKS_TAGS_TO_SKIP'] = 'extraContext'
import mlflow
logging.getLogger('mlflow').setLevel(logging.ERROR)

mlflow.set_registry_uri('databricks-uc')
URI_MODELO = f'models:/{MODELO_REGISTRADO}@champion'

modelo   = mlflow.sklearn.load_model(URI_MODELO)
info     = mlflow.models.get_model_info(URI_MODELO)
features = info.signature.inputs.input_names()
versao   = mlflow.MlflowClient().get_model_version_by_alias(MODELO_REGISTRADO, 'champion').version

print(f'Modelo: {MODELO_REGISTRADO} · versão {versao} · {len(features)} features')

## 4. Predição e ranking

In [0]:
X = dados[features].copy()
NUMERICAS = [c for c in features if pd.api.types.is_numeric_dtype(X[c])]
X[NUMERICAS] = X[NUMERICAS].astype('float64')       # mesmo tipo usado no treino

ranking = (dados[['dtRef', 'IdCliente']]
           .assign(prob_churn=modelo.predict_proba(X)[:, 1],      # coluna 1 = classe churn
                   versao_modelo=str(versao))
           .sort_values('prob_churn', ascending=False)
           .reset_index(drop=True))
ranking['posicao'] = ranking.index + 1

print(f"Probabilidade média de churn: {ranking['prob_churn'].mean():.1%} · "
      f"acima de 50%: {(ranking['prob_churn'] > 0.5).sum()} usuários")
ranking.head(10)

## 5. Salvar
Todas as probabilidades vão para a tabela `predicoes_churn`, uma safra por vez (`replaceWhere`), formando o histórico de predições. O top 50 também sai em CSV, para a ação de retenção.

In [0]:
TABELA_PREDICOES = f'{SCHEMA}.predicoes_churn'

escrita = spark.createDataFrame(ranking).write.format('delta').mode('overwrite')
if spark.catalog.tableExists(TABELA_PREDICOES):
    escrita = escrita.option('replaceWhere', f"dtRef = DATE'{DT_PREDICAO}'")
escrita.saveAsTable(TABELA_PREDICOES)

top50 = ranking.head(50).set_index('posicao')
top50.to_csv(f'{TABLE_DIR}/top50_churn_{DT_PREDICAO}.csv')
print(f'{TABELA_PREDICOES}: safra {DT_PREDICAO} gravada ({len(ranking)} usuários) · top 50 em output/tables/top50_churn_{DT_PREDICAO}.csv')
top50

## 6. Conferência
Jul/26 já tem o alvo conhecido e o modelo, treinado até mai/26, nunca viu esse mês. Por isso dá para conferir a predição: quantos dos 50 usuários de maior risco realmente deixaram de participar nos 28 dias seguintes, comparado com o churn médio do mês.

In [0]:
abt_mes = (spark.table(TABELA_ABT)
           .filter(f"dtRef = DATE'{DT_PREDICAO}'")
           .select('IdCliente', 'churn')
           .toPandas())

conf = top50.merge(abt_mes, on='IdCliente', how='left')
print(f"Top 50 de {mes(DT_PREDICAO)}: {conf['churn'].mean():.0%} saíram de fato "
      f"| churn médio do mês: {abt_mes['churn'].mean():.0%} "
      f"| ganho: {conf['churn'].mean() / abt_mes['churn'].mean():.1f}x")

> **Resultado:** 88% do top 50 de jul/26 saiu de fato, contra 45% de churn médio no mês: **2,0x** mais acerto do que escolher 50 usuários ao acaso. É um segundo mês fora do treino com o mesmo comportamento do out-of-time (jun/26: 80% contra 44%, 1,8x), o que reforça que o ranking se mantém no tempo.